In [70]:
import pandas as pd
from sentence_transformers import SentenceTransformer
import numpy as np
from bertopic import BERTopic
from sklearn.linear_model import LinearRegression
import json

# Loading Embeddings

In [71]:
embeddings = np.load("embeddings.npy")

df = pd.read_parquet("documents.parquet")

# Model Decision

In [72]:
from sklearn.feature_extraction.text import CountVectorizer
from gensim.corpora import Dictionary
from gensim.models import CoherenceModel

vectorizer = CountVectorizer(stop_words="english")
analyzer = vectorizer.build_analyzer()

In [73]:
tokenized_docs  = [analyzer(doc) for doc in df['text'].to_list()]

In [74]:
def compute_coherence(topic_model, docs, sample_size=None, coherence_type="c_v"):
    """
    Faster coherence computation for BERTopic
    """

    # 🔹 Sample kullan (çok önerilir)
    if sample_size is not None:
        docs = docs[:sample_size]

    # 🔹 Tokenize
    vectorizer = CountVectorizer(stop_words="english")
    analyzer = vectorizer.build_analyzer()
    tokenized_docs = [analyzer(doc) for doc in docs]

    # 🔹 Dictionary
    dictionary = Dictionary(tokenized_docs)

    # 🔹 Topic words al
    topics = topic_model.get_topics()
    topic_words = [
        [word for word, _ in topics[topic_id]]
        for topic_id in topics
        if topic_id != -1
    ]

    # 🔹 Coherence hesapla
    coherence_model = CoherenceModel(
        topics=topic_words,
        texts=tokenized_docs,
        dictionary=dictionary,
        coherence=coherence_type
    )

    return coherence_model.get_coherence()

In [75]:
docs = df["text"].tolist()

results = []
models = {}
topics_dict = {} 
for min_size in [20, 50, 100]:

    model = BERTopic(
        min_topic_size=min_size,
        calculate_probabilities=False,
        verbose=True
    )

    topics, _ = model.fit_transform(docs, embeddings)

    score = compute_coherence(
        model,
        docs,
        sample_size=5000  # hız için sample
    )
    topic_count = len(model.get_topic_info()) - 1 
    results.append((min_size, score, topic_count))
    models[min_size] = model
    topics_dict[min_size] = topics

print(results)

2026-03-12 21:40:15,228 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-03-12 21:40:53,497 - BERTopic - Dimensionality - Completed ✓
2026-03-12 21:40:53,500 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-03-12 21:41:03,414 - BERTopic - Cluster - Completed ✓
2026-03-12 21:41:03,444 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-03-12 21:41:26,337 - BERTopic - Representation - Completed ✓
2026-03-12 21:42:38,484 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-03-12 21:43:17,945 - BERTopic - Dimensionality - Completed ✓
2026-03-12 21:43:17,948 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-03-12 21:43:27,417 - BERTopic - Cluster - Completed ✓
2026-03-12 21:43:27,442 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-03-12 21:43:51,582 - BERTopic - Representation - Completed ✓
2026-03-12 21:44:30,340 - BERTopic - D

[(20, np.float64(0.4384609030429912), 706), (50, np.float64(0.46672322673388006), 355), (100, np.float64(0.4958853944517942), 208)]


In [76]:
df_results = pd.DataFrame(results, columns=["min_topic_size", "coherence", "topic_count"])
df_results

,min_topic_size,coherence,topic_count
0,20,0.438461,706
1,50,0.466723,355
2,100,0.495885,208


In [77]:
model = models[100]
topics = topics_dict[100]
reduced_model = model.reduce_topics(docs, nr_topics=100)
reduced_topics, _ = reduced_model.transform(docs, embeddings)

2026-03-12 21:46:21,728 - BERTopic - Topic reduction - Reducing number of topics
2026-03-12 21:46:21,809 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-03-12 21:46:43,373 - BERTopic - Representation - Completed ✓
2026-03-12 21:46:43,402 - BERTopic - Topic reduction - Reduced number of topics from 209 to 100
2026-03-12 21:46:46,956 - BERTopic - Dimensionality - Reducing dimensionality of input embeddings.
2026-03-12 21:46:47,369 - BERTopic - Dimensionality - Completed ✓
2026-03-12 21:46:47,370 - BERTopic - Clustering - Approximating new points with `hdbscan_model`
2026-03-12 21:47:07,278 - BERTopic - Cluster - Completed ✓


In [78]:
#model = BERTopic(
#    min_topic_size=100,
#    calculate_probabilities=False,
#    verbose=True,
#    nr_topics=100)

# Trend Analysis

In [79]:
df["topic"] = reduced_topics
df["month"] = df["published"].dt.to_period("M")

trend = (
    df[df.topic != -1] # -1 means outliers, we ignore them
    .groupby(["month", "topic"])
    .size()
    .unstack(fill_value=0)
)

trend_share = trend.div(trend.sum(axis=1), axis=0)

In [80]:
def compute_trend_scores(trend_share, window=12, min_share=0.001):
    scores = {}

    recent = trend_share.tail(window)
    x = np.arange(len(recent)).reshape(-1, 1)

    mean_share = recent.mean()
    valid_topics = mean_share[mean_share > min_share].index

    for topic in valid_topics:
        y = recent[topic].values

        if y.sum() == 0:
            continue

        model = LinearRegression()
        model.fit(x, y)

        scores[topic] = model.coef_[0]

    return pd.Series(scores).sort_values(ascending=False)

def compute_log_trend_scores(trend_share, window=12, min_share=0.001):
    scores = {}

    recent = trend_share.tail(window)
    x = np.arange(len(recent)).reshape(-1, 1)

    mean_share = recent.mean()
    valid_topics = mean_share[mean_share > min_share].index

    for topic in valid_topics:
        y = recent[topic].values



        y_log = np.log(y + 1e-6)

        model = LinearRegression()
        model.fit(x, y_log)

        scores[topic] = model.coef_[0]

    return pd.Series(scores).sort_values(ascending=False)



In [81]:
log_trend_scores_12 = compute_log_trend_scores(trend_share, window=12)

print("🔥 Emerging topics:")
print(log_trend_scores_12.head(10))

print("\n📉 Declining topics:")
print(log_trend_scores_12.tail(10))

🔥 Emerging topics:
83    0.279847
42    0.085298
81    0.084425
50    0.077901
77    0.065256
19    0.060753
76    0.043227
39    0.043193
48    0.043040
2     0.037452
dtype: float64

📉 Declining topics:
91   -0.279335
80   -0.280208
87   -0.282097
74   -0.292999
89   -0.312418
73   -0.317371
75   -0.317826
52   -0.331803
32   -0.333343
67   -0.531635
dtype: float64


In [82]:
log_trend_scores_24 = compute_log_trend_scores(trend_share, window=24)

print("🔥 Emerging topics:")
print(log_trend_scores_24.head(10))

print("\n📉 Declining topics:")
print(log_trend_scores_24.tail(10))

🔥 Emerging topics:
77    0.079695
19    0.062127
2     0.061306
60    0.056120
42    0.043484
78    0.030466
76    0.024099
59    0.023169
36    0.021311
48    0.021207
dtype: float64

📉 Declining topics:
87   -0.050625
73   -0.059959
40   -0.065599
94   -0.084670
74   -0.084740
75   -0.087068
97   -0.091594
52   -0.121843
32   -0.141625
67   -0.173194
dtype: float64


In [83]:
len(log_trend_scores_12), len(log_trend_scores_24)

(88, 90)

#  Visualization

In [84]:
import plotly.express as px
from sklearn.preprocessing import MinMaxScaler


In [85]:
def clean_label(name):
    parts = name.split("_")[2:]  # topic_x_ kısmını at
    return " ".join(parts[:3])   # ilk 3 keyword yeterli

def generate_label(topic_id, top_n=3):
    words = [word for word, _ in reduced_model.get_topic(topic_id)]
    return " / ".join(words[:top_n])

def assign_quadrant(row):
    if row["slope_24m"] > 0 and row["slope_12m"] > 0:
        return "🚀 Strong & Accelerating"
    
    elif row["slope_24m"] > 0 and row["slope_12m"] < 0:
        return "📈 Strong but Slowing"
    
    elif row["slope_24m"] < 0 and row["slope_12m"] > 0:
        return "🌱 Emerging"
    
    else:
        return "📉 Declining"

In [86]:
topics_df  = reduced_model.get_topic_info()
topics_df = topics_df[topics_df.Topic != -1]

In [87]:
topics_df["label"] = topics_df["Name"].apply(clean_label)
topics_df["label"] = topics_df["label"].str.capitalize()

In [88]:
slope_df =pd.DataFrame({
    "slope_12m": log_trend_scores_12,
    "slope_24m": log_trend_scores_24
})
slope_df["Topic"] = slope_df.index

topics_df = topics_df.merge(slope_df, on="Topic")
topics_df.set_index("Topic", inplace=True)


In [89]:
topic_map = topics_df[["label"]].to_dict()["label"]

In [90]:
topics_df["category"] = topics_df.apply(assign_quadrant, axis=1)
topics_df["acceleration"] = topics_df["slope_12m"] - topics_df["slope_24m"]

In [91]:
topics_df["growth_score"] = (topics_df["slope_12m"] * np.log1p(topics_df["Count"]))
scaler = MinMaxScaler()
topics_df["growth_size"] = scaler.fit_transform(topics_df[["growth_score"]])
topics_df["growth_size"] = topics_df["growth_size"] * 40 + 10

In [92]:
mn = MinMaxScaler()
topics_df[["scaled_acc", "scaled_growth_size"]] = mn.fit_transform(topics_df[["acceleration", "growth_size"]])
topics_df["scaled_acc"].fillna(0, inplace=True)
topics_df["scaled_growth_size"].fillna(0, inplace=True)

In [93]:
topics_df.head()

,Count,Name,Representation,Representative_Docs,label,slope_12m,slope_24m,category,acceleration,growth_score,growth_size,scaled_acc,scaled_growth_size
Topic,,,,,,,,,,,,,
0,6857,0_speech_audio_asr_music,"[speech, audio, asr, music, recognition, speak...",[Continual Speech Learning with Fused Speech F...,Audio asr music,-0.003264,-0.002632,📉 Declining,-0.000633,-0.028834,37.086525,0.553724,0.677163
1,5844,1_explanations_preference_models_to,"[explanations, preference, models, to, the, of...",[Can Large Language Models Be an Alternative t...,Preference models to,-0.015190,-0.018486,📉 Declining,0.003296,-0.131745,36.163802,0.559804,0.654095
2,5090,2_reasoning_mathematical_cot_llms,"[reasoning, mathematical, cot, llms, models, o...",[Towards Reasoning in Large Language Models: A...,Mathematical cot llms,0.037452,0.061306,🚀 Strong & Accelerating,-0.023853,0.319663,40.211261,0.517789,0.755282
3,4719,3_driving_robot_autonomous_manipulation,"[driving, robot, autonomous, manipulation, tra...",[On the Road with GPT-4V(ision): Early Explora...,Robot autonomous manipulation,-0.003723,0.002903,📈 Strong but Slowing,-0.006626,-0.031496,37.062659,0.544449,0.676566
4,3998,4_retrieval_rag_question_knowledge,"[retrieval, rag, question, knowledge, answerin...",[G-RAG: Knowledge Expansion in Material Scienc...,Rag question knowledge,-0.032564,-0.013784,📉 Declining,-0.018780,-0.270083,34.923419,0.525639,0.623085


In [94]:
topics_df["label"].nunique()

90

In [95]:
def get_topic_words(topic_id, top_n=10, model=reduced_model):
    
    words = model.get_topic(topic_id)
    
    words = words[:top_n]
    
    terms = [w[0] for w in words]
    scores = [w[1] for w in words]
    
    return terms, scores

In [96]:
topic_words = {}
for topic_id in topics_df.index:
    terms, scores = get_topic_words(topic_id)
    topic_words[topic_id] = (terms, scores)  

In [97]:
with open('topic_words.json', 'w') as f:
    json.dump(topic_words, f)

In [98]:
topics_df.to_csv("topics_trend_analysis.csv")
trend_share.to_csv("trend_share.csv")


In [99]:
top_topics = topics_df["slope_12m"].sort_values(ascending=False).head(10).index

df_plot = trend_share[top_topics]
df_plot.index = df_plot.index.to_timestamp()
df_plot.columns = df_plot.columns.map(topic_map)

fig = px.line(
    df_plot,
    x=df_plot.index,
    y=df_plot.columns,
    title="Trend Share Over Time 12-Month Slope"
)
fig.update_layout(
    width=1200,
    height=600
)
fig.show()

In [100]:
top_topics = topics_df["slope_24m"].sort_values(ascending=False).head(10).index

df_plot = trend_share[top_topics]
df_plot.index = df_plot.index.to_timestamp()
df_plot.columns = df_plot.columns.map(topic_map)

fig = px.line(
    df_plot,
    x=df_plot.index,
    y=df_plot.columns,
    title="Trend Share Over Time 24-Month Slope"
)
fig.update_layout(
    width=1200,
    height=600
)
fig.show()

In [101]:
fig = px.scatter(
    topics_df,
    x="slope_24m",
    y="slope_12m",
    color="category",
    size="Count",
    hover_name="label",

    title="AI-Powered Research Trend Quadrant"
)

fig.add_vline(x=0, line_dash="dash")
fig.add_hline(y=0, line_dash="dash")

fig.update_layout(
    xaxis_title="24M Growth Rate",
    yaxis_title="12M Growth Rate",
    template="plotly_white"
)

fig.show()

In [102]:
top_topics = topics_df[(topics_df["slope_12m"] > 0)].sort_values(by="acceleration", ascending=False).head(10).index
topics_df = topics_df.loc[top_topics]
    
fig = px.scatter(
    topics_df,
    x="slope_24m",
    y="slope_12m",
    color="category",
    size="Count",
    hover_name="label",
    title="AI-Powered Research Trend Quadrant",
)


fig.add_vline(x=0, line_dash="dash")
fig.add_hline(y=0, line_dash="dash")

fig.update_layout(
    xaxis_title="24M Growth Rate",
    yaxis_title="12M Growth Rate",
    template="plotly_white"
)

fig.show()

In [103]:
df_plot = topics_df.dropna()
df_plot["category"] = df_plot.apply(assign_quadrant, axis=1)
fig = px.scatter(
    df_plot,
    x="slope_24m",
    y="slope_12m",
    size="growth_size",
    color="category",
    hover_name="label",
    title="AI-Powered Research Trend Intelligence Quadrant"
)

fig.add_vline(x=0, line_dash="dash")
fig.add_hline(y=0, line_dash="dash")

fig.update_layout(template="plotly_white")

fig.show()

In [104]:
top10_growth = (topics_df.sort_values("growth_score", ascending=False).head(10).index)

topics_df_plot = topics_df.loc[top10_growth]
fig = px.scatter(
    topics_df_plot,
    x="slope_24m",
    y="slope_12m",
    size="growth_size",
    color="category",
    hover_name="label",
    title="AI-Powered Research Trend Intelligence Quadrant"
)

fig.add_vline(x=0, line_dash="dash")
fig.add_hline(y=0, line_dash="dash")

fig.update_layout(template="plotly_white")

fig.show()

In [105]:
impact_df = topics_df[(topics_df['slope_12m'] > 0)&
                    (topics_df["acceleration"] > 0)].copy()

impact_df = impact_df.sort_values('growth_score', ascending=False)

top10_impact = impact_df.head(10)

In [106]:
top10_impact.to_csv("top10_impact_topics.csv")

In [107]:
raw_data = pd.read_csv("arxiv_abstracts_final.csv").drop_duplicates()

In [108]:
raw_data.head()

,paper_id,title,abstract,published,categories
0,http://arxiv.org/abs/2101.00117v1,Multi-task Retrieval for Knowledge-Intensive T...,Retrieving relevant contexts from a large corp...,2021-01-01,['cs.CL']
1,http://arxiv.org/abs/2101.00121v2,WARP: Word-level Adversarial ReProgramming,Transfer learning from pretrained language mod...,2021-01-01,['cs.CL']
2,http://arxiv.org/abs/2101.00123v2,Intent Classification and Slot Filling for Pri...,Understanding privacy policies is crucial for ...,2021-01-01,['cs.CL']
3,http://arxiv.org/abs/2101.00124v3,Discourse-level Relation Extraction via Graph ...,The ability to capture complex linguistic stru...,2021-01-01,"['cs.CL', 'cs.AI', 'cs.LG']"
4,http://arxiv.org/abs/2101.00130v1,Sensei: Self-Supervised Sensor Name Segmentation,"A sensor name, typically an alphanumeric strin...",2021-01-01,['cs.CL']


In [109]:
rep_docs = reduced_model.get_representative_docs()

rows = []

for topic_id, docs in rep_docs.items():
    for rant, text in enumerate(docs, start=1):
        paper_id = df[df['text'] == text]['paper_id']
        paper_data = raw_data[raw_data['paper_id'] == paper_id.values[0]]
        rows.append({
            "topic_id": topic_id,
            "rank": rant,
            "text": text[:200],
            "title": paper_data['title'].values[0],
            "abstract": paper_data['abstract'].values[0],
            "paper_id": paper_data['paper_id'].values[0],
            "published": paper_data['published'].values[0]
        })
rep_docs_df = pd.DataFrame(rows)
rep_docs_df.to_csv("representative_docs.csv", index=False)